# 03 — Dataset Exploration: Know Your Data Before You Train

## 1. Learning Goal

Before any model: **look at the data**. You will learn, and show with code:

1. **Dataset, label, class**: what they are in a sign project
2. How to **record your own** small sign dataset (recommended!)
3. **Class distribution** and **class imbalance**
4. **Sample images, image sizes, brightness**
5. **Train / validation / test** split
6. **Data leakage**: the most common way to fool yourself with a great-looking accuracy

## 2. Concept

```text
Dataset  = many examples
Example  = image  +  label
Label    = the answer we want (e.g. "A", "B", "L")
Class    = one possible label (the set of classes: A, B, L, ...)
```

```text
Train       : the model learns from these
Validation  : we use these to compare models and settings during development
Test        : locked away. Used ONCE at the very end for an honest score
```

## 3. Why do we need this?

> "Garbage in, garbage out." If your classes are unbalanced, your images are all from one lighting condition, or your validation images are almost copies of training images, your accuracy is **meaningless**, however high it looks.

In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Works whether you start Jupyter in the project folder or inside notebooks/
ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA = ROOT / "data"
MODELS = ROOT / "models"
DATA.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

def show_bgr(img, title="", size=6):
    plt.figure(figsize=(size, size))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # OpenCV is BGR, Matplotlib wants RGB
    plt.title(title)
    plt.axis("off")
    plt.show()

def describe(image, name="image"):
    print(f"--- {name} ---")
    print("shape :", image.shape, "| dtype:", image.dtype, "| min:", image.min(), "| max:", image.max())

In [ ]:
import pandas as pd
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split

SIGNS_DIR = DATA / "signs"         # layout:  data/signs/<label>/<session>_<number>.jpg
print("Looking for images in:", SIGNS_DIR.resolve())

---
# Part 1 — Get a dataset

## Option A (recommended): record your own with the webcam

**Why your own?** It matches the exact camera, room and hand you will use later. You also control **sessions**.

**Session** = one recording round. Between sessions change something: time of day or lamp, background, distance to the camera, angle. **Use a new session name each time** (`s1`, `s2`, `s3`, ...).

```text
data/signs/
   A/   s1_000.jpg  s1_001.jpg ...  s2_000.jpg ...
   B/   s1_000.jpg ...
   L/   ...
```

Plan for the first dataset:

- **5 signs** that look clearly different (for example the ASL letters A, B, L, Y and 5/open hand)
- **at least 3 sessions**, **60+ images per sign per session**
- while recording, slowly **move and rotate your hand a little**

How the recorder works: press **SPACE** to start, it saves every 3rd frame until it has enough, press **q** to quit. Frames are saved **mirrored** (what you see on screen), because the live app will be mirrored too.

In [ ]:
def record_sign(label, session, n_images=60, every_n_frames=3, camera=0):
    out_dir = SIGNS_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)

    if any(out_dir.glob(f"{session}_*.jpg")):
        print(f"Session '{session}' already exists for '{label}'. Use a new session name.")
        return

    cap = cv2.VideoCapture(camera)
    if not cap.isOpened():
        print("Could not open the webcam.")
        return

    saved, frame_count, recording = 0, 0, False
    while saved < n_images:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)                       # mirror: save what you see
        frame_count += 1

        if recording and frame_count % every_n_frames == 0:
            cv2.imwrite(str(out_dir / f"{session}_{saved:03d}.jpg"), frame)
            saved += 1

        preview = frame.copy()
        status = f"{label} / {session}: {saved}/{n_images}" if recording else "SPACE = start, q = quit"
        cv2.putText(preview, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow("record", preview)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(" "):
            recording = True
        elif key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"saved {saved} images to {out_dir}")

RECORD = False          # set True and edit the lines below to record
if RECORD:
    record_sign("A", "s1")

To record everything, repeat for each sign and each session, for example:

```python
for sign in ["A", "B", "L", "Y", "5"]:
    record_sign(sign, "s1")      # then later "s2", "s3" in a different lighting / place
```

## Option B: a public dataset (for practice or extra data)

Download **ASL Alphabet** from Kaggle and unzip so that you have `data/signs/<Letter>/*.jpg`. Warning: its file names have **no session information** and many images are near-duplicates from the same person. Everything below still runs, but see the leakage section to understand why such datasets give inflated scores. Always check the dataset's licence before using it.

---
# Part 2 — List what we have

In [ ]:
def list_dataset(root):
    rows = []
    for class_dir in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        for p in sorted(class_dir.glob("*")):
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                session = p.stem.split("_")[0] if "_" in p.stem else "unknown"
                rows.append({"path": str(p), "label": class_dir.name, "session": session})
    return pd.DataFrame(rows)

if not SIGNS_DIR.exists():
    raise FileNotFoundError("No data/signs folder yet. Record some images first (Option A) or download one (Option B).")

df = list_dataset(SIGNS_DIR)
print("total images:", len(df))
print("classes     :", sorted(df.label.unique()))
print("sessions    :", sorted(df.session.unique()))
df.head()

---
# Part 3 — Class distribution and imbalance

## Concept

If 90% of your images are class "A", a "model" that always answers "A" is 90% accurate and completely useless. **Imbalance** hides this. Always plot counts first.

In [ ]:
counts = df.label.value_counts().sort_index()
print(counts)

plt.figure(figsize=(7, 3.5))
plt.bar(counts.index, counts.values)
plt.title("Images per class")
plt.ylabel("count")
plt.show()

ratio = counts.max() / counts.min()
print(f"imbalance ratio (largest / smallest class): {ratio:.2f}")
print("majority-class baseline accuracy:", round(counts.max() / counts.sum(), 3))

In [ ]:
# Per class AND per session
pd.crosstab(df.label, df.session)

### What to observe

- The **majority-class baseline** is the accuracy of a model that always predicts the most common class. Every real model must beat this number.
- The class × session table shows if one session or class is under-represented.

### Challenge 1

If one class had 10× fewer images than the others, name two ways to deal with it (hint: collect more, or weight/oversample).

---
# Part 4 — Look at the images

Never trust numbers alone. Open the images.

In [ ]:
rng = np.random.default_rng(0)
classes = sorted(df.label.unique())
per_class = 4

fig, axes = plt.subplots(len(classes), per_class, figsize=(2.4 * per_class, 2.4 * len(classes)))
axes = np.atleast_2d(axes)
for r, label in enumerate(classes):
    subset = df[df.label == label]
    picks = subset.sample(min(per_class, len(subset)), random_state=0)
    for c in range(per_class):
        ax = axes[r, c]
        ax.axis("off")
        if c < len(picks):
            row = picks.iloc[c]
            ax.imshow(cv2.cvtColor(cv2.imread(row.path), cv2.COLOR_BGR2RGB))
            ax.set_title(f"{row.label} ({row.session})", fontsize=9)
plt.tight_layout()
plt.show()

### What to observe

- Is the hand fully visible? Are there bad frames (blur, hand out of view, wrong sign)? **Delete bad images from the folder** and re-run.
- Do different sessions look different (light, background)? Good: variety makes the model robust.

---
# Part 5 — Image sizes and brightness

In [ ]:
sample = df.sample(min(300, len(df)), random_state=0).copy()
sizes, brightness = [], []
for p in sample.path:
    im = cv2.imread(p)
    sizes.append(im.shape[:2])
    brightness.append(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY).mean())
sample["height"] = [s[0] for s in sizes]
sample["width"] = [s[1] for s in sizes]
sample["brightness"] = brightness

print("distinct image sizes (h, w):", sample[["height", "width"]].drop_duplicates().values.tolist()[:10])

sessions = sorted(sample.session.unique())
plt.figure(figsize=(7, 3.5))
plt.boxplot([sample[sample.session == s].brightness for s in sessions])
plt.xticks(range(1, len(sessions) + 1), sessions)
plt.ylabel("mean brightness (0 = black, 255 = white)")
plt.title("Brightness per session")
plt.show()

### What to observe

- If all images have **one size** that is great; otherwise we must resize later.
- If sessions have different brightness your data has real variety. If every session looks the same, the model may fail under a different lamp.

---
# Part 6 — Splitting: train / validation / test, and DATA LEAKAGE

## Concept

When we record at 30 frames per second, **consecutive frames are almost identical**. If we shuffle all frames randomly into train and validation, nearly every validation image has a near-twin in the training set. The model only has to *remember*, not *understand*, and the accuracy looks fantastic. When you test with a new lighting setup, it collapses.

```text
RANDOM split:   train: s1_001  s1_003 ...      validation: s1_002 (twin of s1_001!)   ← LEAKAGE
SESSION split:  train: s1, s2                  validation: s3 (never seen)            ← honest
```

## Experiment: measure the leakage

For each validation image, find the **distance to its nearest training image** (tiny 16×16 grayscale versions). Small distance = near-duplicate = easy.

In [ ]:
def tiny_vector(path):
    gray = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2GRAY)
    return cv2.resize(gray, (16, 16)).astype(np.float32).flatten()

def median_nearest_distance(train_paths, val_paths):
    train_vecs = np.array([tiny_vector(p) for p in train_paths])
    val_vecs = np.array([tiny_vector(p) for p in val_paths])
    nearest = pairwise_distances(val_vecs, train_vecs).min(axis=1)
    return float(np.median(nearest))

# 1) Random split (the leaky way)
rand_train, rand_val = train_test_split(df, test_size=0.3, stratify=df.label, random_state=0)

# 2) Session split: hold out whole sessions
all_sessions = sorted(df.session.unique())
print("sessions available:", all_sessions)

In [ ]:
if len(all_sessions) >= 3 and "unknown" not in all_sessions:
    val_session = all_sessions[-1]
    grp_train = df[df.session != val_session]
    grp_val = df[df.session == val_session]

    d_random = median_nearest_distance(rand_train.path, rand_val.path)
    d_session = median_nearest_distance(grp_train.path, grp_val.path)

    print(f"median distance to nearest training image")
    print(f"  random split : {d_random:8.1f}   <- validation images have near-twins")
    print(f"  session split: {d_session:8.1f}   <- validation images are genuinely new")

    plt.bar(["random split", "session split"], [d_random, d_session], color=["tab:red", "tab:green"])
    plt.ylabel("distance to nearest training image (smaller = easier)")
    plt.show()
else:
    print("You need at least 3 recording sessions (and session names in the file names) for this experiment.")
    print("Record more sessions with different light/background, then re-run.")

### What to observe

- The **random split** has a much smaller distance: its validation set is "easy" because of near-duplicate frames.
- The **session split** is the honest one. Expect lower accuracy numbers, and trust them.

## The final split we will use

```text
train      : most sessions
validation : 1 session   (to compare models / settings)
test       : 1 session   (locked away until the very end)
```

In [ ]:
def split_by_session(df, seed=0):
    sessions = sorted(df.session.unique())
    if len(sessions) < 3 or "unknown" in sessions:
        print("WARNING: fewer than 3 named sessions, falling back to a RANDOM split (risk of leakage!).")
        train, rest = train_test_split(df, test_size=0.3, stratify=df.label, random_state=seed)
        val, test = train_test_split(rest, test_size=0.5, stratify=rest.label, random_state=seed)
        out = df.copy()
        out["split"] = "train"
        out.loc[val.index, "split"] = "val"
        out.loc[test.index, "split"] = "test"
        return out

    rng = np.random.default_rng(seed)
    shuffled = list(rng.permutation(sessions))
    test_s, val_s = shuffled[0], shuffled[1]
    out = df.copy()
    out["split"] = "train"
    out.loc[out.session == val_s, "split"] = "val"
    out.loc[out.session == test_s, "split"] = "test"
    print("validation session:", val_s, "| test session:", test_s)
    return out

splits = split_by_session(df)
print(pd.crosstab(splits.label, splits.split))

splits.to_csv(DATA / "splits.csv", index=False)
print("saved", DATA / "splits.csv")

### Challenge 2

1. Change the seed in `split_by_session`. Which session is now the test session? Why might results change a little depending on which session is held out?
2. Why must the **test set stay locked** until the end? What goes wrong if you look at it every time you change a setting?

---
# 9. Results

```text
data/signs/<label>/<session>_<n>.jpg
        ↓
table (path, label, session)
        ↓
checked: class balance, samples, sizes, brightness
        ↓
split BY SESSION → data/splits.csv  (train / val / test)
```

# 10. What did we learn?

1. A dataset is images + labels; classes are the possible labels.
2. Plot class counts first. Know the **majority-class baseline**.
3. Look at real images; remove bad ones.
4. Near-duplicate frames cause **data leakage** with random splits.
5. Split by **session** (or by person) so validation/test are really new.
6. Keep the **test set locked** until the end.

# 11. Questions to think about

1. If you collected all your data in one room, what would you expect when a friend tests it in theirs?
2. For a public dataset made by 5 people, how would you split to avoid leakage?
3. Why is a model with 99% validation accuracy on a leaky split potentially worse than one with 85% on an honest split?
4. Which would you rather collect: 1000 frames in one session or 300 frames in each of 3 sessions? Why?

# 12. Mini challenge

Add a **fourth session** with a very different setup (dark room, or hand much further away). Re-run the split and the leakage experiment. Does the session-split distance change?

---
## Experiment log

```text
EXP-003
Notebook : 03_dataset_exploration
Dataset  : my own signs, N classes, S sessions
Observed : class imbalance ratio =   | leakage distances: random =  session =
Conclusion :
```

**Next:** `06_baseline_classifier.ipynb`.